# UFO Sighting Country Predictor — Model Training

使用逻辑回归模型，根据目击持续时间、纬度、经度预测 UFO 在哪个国家被目击。

**预测目标国家：** Australia / Canada / Germany / UK / US

## Step 1：加载数据

In [ ]:
import pandas as pd
import numpy as np

ufos = pd.read_csv('./data/ufos.csv')
print(f'原始数据行数：{len(ufos)}')
ufos.head()

## Step 2：数据清洗与特征选取

In [ ]:
ufos = pd.DataFrame({
    'Seconds': ufos['duration (seconds)'],
    'Country': ufos['country'],
    'Latitude': ufos['latitude'],
    'Longitude': ufos['longitude']
})

print('原始国家值：', ufos['Country'].unique())

# 去除空值，只保留 1~60 秒的目击记录
ufos.dropna(inplace=True)
ufos = ufos[(ufos['Seconds'] >= 1) & (ufos['Seconds'] <= 60)]

print(f'清洗后数据行数：{len(ufos)}')
ufos.info()

## Step 3：标签编码（国家名 → 数字）

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
ufos['Country'] = le.fit_transform(ufos['Country'])

print('编码映射：', list(enumerate(le.classes_)))
# 0=au(Australia), 1=ca(Canada), 2=de(Germany), 3=gb(UK), 4=us(US)
ufos.head()

## Step 4：划分训练集与测试集

In [ ]:
from sklearn.model_selection import train_test_split

X = ufos[['Seconds', 'Latitude', 'Longitude']]
y = ufos['Country']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

print(f'训练集：{len(X_train)} 条，测试集：{len(X_test)} 条')

## Step 5：训练逻辑回归模型

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f'准确率：{accuracy_score(y_test, predictions):.4f}')
print()
print(classification_report(y_test, predictions,
      target_names=['Australia','Canada','Germany','UK','US']))

## Step 6：保存模型（Pickle 序列化）

In [ ]:
import pickle

with open('ufo-model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('模型已保存为 ufo-model.pkl')

# 验证：加载并预测
loaded_model = pickle.load(open('ufo-model.pkl', 'rb'))
countries = ['Australia', 'Canada', 'Germany', 'UK', 'US']
test_input = [[50, 44, -12]]  # 50秒，纬度44，经度-12
pred = loaded_model.predict(test_input)[0]
print(f'验证预测：输入 {test_input[0]} → {countries[pred]}')